# 🫁 Chest X-Ray Attention Training - All Models

**Training all 7 attention-enhanced models on Kaggle GPU (P100)**

### Models:
1. ResNet50 Baseline
2. ResNet50 + SE
3. ResNet50 + CBAM ⭐ (Best expected)
4. ResNet50 + ECA
5. DenseNet121 Baseline
6. DenseNet121 + SE
7. DenseNet121 + CBAM

**Dataset:** Chest X-Ray (Pneumonia, Covid-19, Tuberculosis, Normal)  
**GPU:** Tesla P100 (16GB)  
**Estimated Time:** ~3-4 hours for all models

---

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers scikit-learn matplotlib seaborn tqdm opencv-python-headless pyyaml -q
print('✅ Dependencies installed')

In [ ]:
# Clone repository
import os
from pathlib import Path
import subprocess

repo_dir = Path('chest-xray-disease-classifier')

if not repo_dir.exists():
    print('🔄 Cloning repository...')
    try:
        # Try HTTPS
        !git clone https://github.com/kinhluan/chest-xray-disease-classifier.git
        print('✅ Cloned via HTTPS')
    except:
        # Fallback: Try git protocol
        print('⚠️  HTTPS failed, trying git://')
        !git clone git://github.com/kinhluan/chest-xray-disease-classifier.git
    
    %cd chest-xray-disease-classifier
    !pip install -e . -q
    print('✅ Repository cloned and installed')
else:
    %cd chest-xray-disease-classifier
    print('✅ Repository already exists')
    
# Verify
print(f'\n📁 Working directory: {os.getcwd()}')
print(f'📂 Files: {len(list(Path(".").glob("*")))} items')

In [ ]:
# Verify GPU
import torch
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download Dataset

In [ ]:
# Download dataset from Kaggle
!kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis -p data/raw --unzip
print('✅ Dataset downloaded')

In [ ]:
# Verify dataset
from pathlib import Path
data_dir = Path('data/raw')

print(f"\n📁 Dataset Structure:")
for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        n_images = len(list(class_dir.glob('*.png'))) + len(list(class_dir.glob('*.jpg')))
        print(f"  {class_dir.name}: {n_images:,} images")

total = sum(len(list(d.glob('*.png'))) + len(list(d.glob('*.jpg'))) 
           for d in data_dir.iterdir() if d.is_dir())
print(f"\n📊 Total: {total:,} images")

## 3. Train All Models

In [ ]:
# Training configuration
MODELS = [
    ('resnet50_baseline', 'resnet', 'resnet50', 'none'),
    ('resnet50_se', 'attention', 'resnet50', 'se'),
    ('resnet50_cbam', 'attention', 'resnet50', 'cbam'),
    ('resnet50_eca', 'attention', 'resnet50', 'eca'),
    ('densenet121_baseline', 'densenet', 'densenet121', 'none'),
    ('densenet121_se', 'attention', 'densenet121', 'se'),
    ('densenet121_cbam', 'attention', 'densenet121', 'cbam'),
]

# Training parameters
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
DATA_DIR = 'data/raw'
OUTPUT_DIR = 'results/models'

print(f"📋 Training Configuration:")
print(f"  Models: {len(MODELS)}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
import subprocess
from datetime import datetime

# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Training loop
results = []

for i, (model_name, model_type, backbone, attention_type) in enumerate(MODELS, 1):
    print(f"\n{'='*60}")
    print(f"🚀 [{i}/{len(MODELS)}] Training: {model_name}")
    print(f"{'='*60}")
    print(f"  Backbone: {backbone}")
    print(f"  Attention: {attention_type}")
    print(f"  Start: {datetime.now().strftime('%H:%M:%S')}")
    
    # Build command
    cmd = [
        'python', 'train.py',
        '--data_dir', DATA_DIR,
        '--model_type', model_type,
        '--model_name', backbone,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--lr', str(LEARNING_RATE),
        '--output_dir', OUTPUT_DIR,
        '--experiment_name', model_name,
        '--scheduler', 'cosine',
        '--use_amp',
        '--seed', '42'
    ]
    
    if model_type == 'attention' and attention_type != 'none':
        cmd.extend(['--attention_type', attention_type])
    
    # Run training
    try:
        result = subprocess.run(cmd, check=True)
        
        # Check if model saved
        model_path = Path(OUTPUT_DIR) / model_name / 'best_model.pth'
        if model_path.exists():
            print(f"\n✅ {model_name} completed successfully!")
            results.append({'model': model_name, 'status': 'success'})
        else:
            print(f"\n⚠️  {model_name} completed but no checkpoint found")
            results.append({'model': model_name, 'status': 'warning'})
            
    except subprocess.CalledProcessError as e:
        print(f"\n❌ {model_name} failed: {e}")
        results.append({'model': model_name, 'status': 'failed'})
    
    print(f"  End: {datetime.now().strftime('%H:%M:%S')}")

# Summary
print(f"\n{'='*60}")
print("📊 TRAINING SUMMARY")
print(f"{'='*60}")
for r in results:
    status_icon = '✅' if r['status'] == 'success' else '⚠️' if r['status'] == 'warning' else '❌'
    print(f"{status_icon} {r['model']}: {r['status']}")

success_count = sum(1 for r in results if r['status'] == 'success')
print(f"\n✅ Successfully trained: {success_count}/{len(MODELS)} models")

## 4. Verify Trained Models

In [ ]:
# List all trained models
from pathlib import Path

models_dir = Path(OUTPUT_DIR)
print(f"\n📁 Trained Models:")
print(f"{'='*60}")

for model_dir in sorted(models_dir.iterdir()):
    if model_dir.is_dir():
        checkpoint = model_dir / 'best_model.pth'
        config = model_dir / 'config.json'
        metrics = model_dir / 'metrics.json'
        
        status = '✅' if checkpoint.exists() else '❌'
        size = f"{checkpoint.stat().st_size / 1e6:.1f} MB" if checkpoint.exists() else 'N/A'
        
        print(f"{status} {model_dir.name:30s} | Size: {size}")

print(f"\n📂 Output directory: {models_dir.absolute()}")

## 5. Next Steps

### Option A: Evaluate on Kaggle
```bash
!python scripts/evaluate_models.py
!python scripts/compare_results.py
!python scripts/visualize_results.py
```

### Option B: Download Results
```bash
# Download all models
!kaggle kernels output luanbhk/chest-xray-attention-training -p ../output
```

### Option C: Continue Training More Models
Re-run this notebook with different hyperparameters.

---

**Training Complete! 🎉**